<a href="https://colab.research.google.com/github/Simranpande/retail-demand-forecasting/blob/main/notebooks/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U kaggle

import os, getpass
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Paste your Kaggle token: ")

Paste your Kaggle token: ··········


In [2]:
!kaggle competitions download -c m5-forecasting-accuracy -p /content/m5
!unzip -o -q /content/m5/m5-forecasting-accuracy.zip -d /content/m5
!ls -lh /content/m5

100% 45.8M/45.8M [00:00<00:00, 75.8MB/s]

total 476M
-rw-r--r-- 1 root root 102K Jun  1  2020 calendar.csv
-rw-r--r-- 1 root root  46M Jun  1  2020 m5-forecasting-accuracy.zip
-rw-r--r-- 1 root root 117M Jun  1  2020 sales_train_evaluation.csv
-rw-r--r-- 1 root root 115M Jun  1  2020 sales_train_validation.csv
-rw-r--r-- 1 root root 5.0M Jun  1  2020 sample_submission.csv
-rw-r--r-- 1 root root 194M Jun  1  2020 sell_prices.csv


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/retail-demand-forecasting/data/"
os.makedirs(BASE + "processed", exist_ok=True)

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

RAW = "/content/m5/"   # Option 2 users: use BASE + "raw/" instead

cal = pd.read_csv(RAW + "calendar.csv", parse_dates=["date"])
prices = pd.read_csv(RAW + "sell_prices.csv")

day_cols = {f"d_{i}": "int16" for i in range(1, 1942)}
sales = pd.read_csv(RAW + "sales_train_evaluation.csv", dtype=day_cols)

print(cal.shape, prices.shape, sales.shape)

(1969, 14) (6841121, 4) (30490, 1947)


In [5]:
sales = sales[sales["store_id"] == "CA_1"].copy()

d_cols = [c for c in sales.columns if c.startswith("d_")]
sales["total_units"] = sales[d_cols].sum(axis=1)
sales = sales.nlargest(100, "total_units").drop(columns="total_units")
print(sales.shape)   # (100, 1947)

(100, 1947)


In [6]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
long = sales.melt(id_vars=id_cols, var_name="d", value_name="units")

cal_keep = cal[["d", "date", "wm_yr_wk", "weekday", "month", "year",
                "event_name_1", "event_type_1", "snap_CA"]]
long = long.merge(cal_keep, on="d", how="left")
long = long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
print(long.shape)

(194100, 17)


In [7]:
before = len(long)
long = long.dropna(subset=["sell_price"]).copy()
print(f"Dropped {before - len(long)} rows (item not yet on sale)")

long["event_name_1"] = long["event_name_1"].fillna("None")
long["event_type_1"] = long["event_type_1"].fillna("None")
long = long.sort_values(["id", "date"]).reset_index(drop=True)

Dropped 8981 rows (item not yet on sale)


In [8]:
print(long.isna().sum())
print("Duplicates:", long.duplicated(subset=["id", "date"]).sum())  # should be 0
print("Date range:", long["date"].min(), "->", long["date"].max())
print("Zero-sales share:", (long["units"] == 0).mean().round(3))
print(long["units"].describe())

id              0
item_id         0
dept_id         0
cat_id          0
store_id        0
state_id        0
d               0
units           0
date            0
wm_yr_wk        0
weekday         0
month           0
year            0
event_name_1    0
event_type_1    0
snap_CA         0
sell_price      0
dtype: int64
Duplicates: 0
Date range: 2011-01-29 00:00:00 -> 2016-05-22 00:00:00
Zero-sales share: 0.156
count    185119.000000
mean         13.048153
std          15.585399
min           0.000000
25%           4.000000
50%           9.000000
75%          17.000000
max         648.000000
Name: units, dtype: float64


In [9]:
long.to_parquet(BASE + "processed/m5_ca1_top100.parquet", index=False)